### Part 1 - Installation and Data


#### Vega-Altair vs Seaborn

While both Vega-Altair and Seaborn are powerful visualization libraries, they have distinct approaches:

| Feature | Seaborn | Vega-Altair |
|---------|---------|-------------|
| **Paradigm** | Matplotlib-based imperative | Grammar of Graphics declarative |
| **Syntax** | Function-based (sns.scatterplot()) | Method chaining (alt.Chart().mark_circle()) |
| **Interactivity** | Limited (requires additional libraries) | Built-in interactive features |
| **Output** | Static plots (primarily) | Interactive web-based visualizations |
| **Learning Curve** | Moderate | Steeper initially, more flexible later |
| **Customization** | High (matplotlib backend) | Very high (Vega-Lite specification) |
| **Web Integration** | Requires conversion | Native web format (JSON) |

Vega-Altair excels at creating interactive, web-ready visualizations with a consistent grammar-based approach.

In [ ]:
!pip -q install altair vega_datasets

In [ ]:
import altair as alt
import pandas as pd
from vega_datasets import data

alt.renderers.enable('colab') # make Altair display in Colab

In [ ]:
cars = data.cars()
cars.head()

In [ ]:
cars = data.cars()
alt.Chart(cars).mark_point().encode(x='Horsepower', # mark_rect
                                    y='Miles_per_Gallon',
                                    color='Origin').properties(width=400, height=300).interactive()


In [ ]:
df = pd.read_csv('WHR_2024.csv')
key_vars = ['Happiness_Score',
            'GDP_per_Capita',
            'Social_Support',
            'Healthy_Life_Expectancy',
            'Freedom']
df[key_vars].describe().round(3)

The dataset includes:
- **Happiness_Score**: Overall happiness score (0-10 scale)
- **GDP_per_Capita**: Economic prosperity indicator
- **Social_Support**: Social network strength
- **Healthy_Life_Expectancy**: Health and longevity measure
- **Freedom**: Freedom to make life choices
- **Generosity**: Generosity level in the country
- **Trust_in_Government**: Perception of corruption (inverted)
- **Dystopia_Residual**: Residual component explaining unexplained variance

### Part 2 - Basic Plotting

In [ ]:
df.columns

In [ ]:
# Calls `.interactive()` to enable pan/zoom interactions on axes.
alt.Chart(df).mark_point().encode(x='GDP_per_Capita',
                                  y='Happiness_Score',
                                  color='Continent').interactive()

In [ ]:
alt.Chart(df).mark_point().encode(x='Continent',
                                  y='Happiness_Score')

In [ ]:
# `average` aggregation. Aggregations include `sum`, `max`, `min`, `count`, etc.
alt.Chart(df).mark_point().encode(x='Continent',
                                  y='average(Happiness_Score)',
                                  color='Continent')

In [ ]:
# Styled scatter with tooltips
alt.Chart(df).mark_point().encode(
    x=alt.X('GDP_per_Capita:Q', # Q-> Quantitative
            title='GDP per Capita'),
    y=alt.Y('Happiness_Score:Q',
            title='Happiness Score'),
    color=alt.Color('Continent',
                    scale=alt.Scale(scheme='viridis')),
    tooltip=['Country:N', # N-> Nominal
             'Happiness_Score:Q',
             'GDP_per_Capita:Q']
).properties(
    title='Relationship between GDP per Capita and Happiness Score',
    width=400,
    height=300
).interactive()

In [ ]:
# Figure 2: Bar Chart - Top 20 Happiest Countries
top_20 = df.head(20)

alt.Chart(top_20).mark_bar().encode(
    x=alt.X('Happiness_Score:Q', title='Happiness Score'),
    y=alt.Y('Country:N', sort='-x', title='Country'),
    color=alt.Color('Continent:N') # tooltip
).properties(
    title='Top 20 Happiest Countries (2024)',
    width=400,
    height=250
).interactive()

### Part 3 - Layering Plots

In [ ]:
# Figure 3. Scatter plot with regression line
# Layers a colored scatter and a red regression line via `transform_regression`
scatter_reg = alt.Chart(df).mark_circle(size=60, opacity=0.7).encode(
    x=alt.X('GDP_per_Capita:Q', title='GDP per Capita'),
    y=alt.Y('Happiness_Score:Q', title='Happiness Score'),
    color=alt.Color('Continent:N', scale=alt.Scale(scheme='category10'))
)

regression = alt.Chart(df).mark_line(color='red').transform_regression(
    'GDP_per_Capita', 'Happiness_Score'
).encode(
    x='GDP_per_Capita:Q',
    y='Happiness_Score:Q'
)

alt.layer(scatter_reg, regression)
# or simply scatter_reg + regression

In [ ]:
# Figure 4: Histogram with Density Overlay
histogram = alt.Chart(df).mark_bar(opacity=0.7).encode(
    x=alt.X('Happiness_Score:Q', bin=alt.Bin(maxbins=20), title='Happiness Score'),
    y=alt.Y('count():Q', title='Count'),
    color=alt.value('steelblue')
)


density = alt.Chart(df).transform_density( # KDE
    'Happiness_Score',
    as_=['Happiness_Score', 'density']
).mark_area(opacity=0.3, color='red').encode(
    x='Happiness_Score:Q',
    y=alt.Y('density:Q', title='Density')
)


# Apply resolve_scale to the layered chart, not individual components
# Prevent the density area from being squashed by the much larger count values
hist_density = (histogram + density).resolve_scale(
    y='independent' #t
).properties(
    title='Distribution of Happiness Scores with Density Overlay',
    width=500,
    height=300
)

hist_density


### Part 4 - Heat Maps

In [ ]:
corr_vars = ['Happiness_Score',
             'GDP_per_Capita',
             'Freedom'] # 3 x 3

df[corr_vars].corr().reset_index().melt('index')

In [ ]:
# Figure 5: Correlation Matrix Heatmap
# Prepare data for correlation matrix
corr_vars = ['Happiness_Score',
             'GDP_per_Capita',
             'Social_Support',
             'Healthy_Life_Expectancy',
             'Freedom',
             'Generosity',
             'Trust_in_Government'] # 7 by 7 matrix

corr_matrix = df[corr_vars].corr().reset_index()
corr_matrix = corr_matrix.melt('index') # wide to long
corr_matrix.columns = ['Variable1', 'Variable2', 'Correlation']


base = alt.Chart(corr_matrix).encode(
    x=alt.X('Variable1:N', title=''),
    y=alt.Y('Variable2:N', title=''))

heatmap = base.mark_rect().encode(
    color=alt.Color('Correlation:Q',
                   scale=alt.Scale(scheme='spectral', domain=[-1, 1])),
    tooltip=['Variable1:N', 'Variable2:N', 'Correlation:Q']
    )

labels = base.mark_text(size=12, fontWeight='bold').encode(
    text=alt.Text('Correlation:Q', format='.2f')
)

(heatmap + labels).properties(
    title='Correlation Matrix of Happiness Factors',
    width=400,
    height=400
)

### Part 5 - Multi-View

In [ ]:
scatter = alt.Chart(df).mark_circle(size=60, opacity=0.7).encode(
    x=alt.X('GDP_per_Capita:Q', title='GDP per Capita'),
    y=alt.Y('Happiness_Score:Q', title='Happiness Score'),
    color=alt.Color('Continent:N', scale=alt.Scale(scheme='category10'))
).properties(
    title='GDP vs Happiness',
    width=250,
    height=200
)

# Box plot by region
boxplot = alt.Chart(df).mark_boxplot().encode(
    x=alt.X('Continent:N', title='Continent'),
    y=alt.Y('Happiness_Score:Q', title='Happiness Score'),
    color=alt.Color('Continent:N', scale=alt.Scale(scheme='category10'))
).properties(
    title='Happiness Distribution by Continent',
    width=250,
    height=200
)

# Combine charts horizontally
dashboard = alt.hconcat(scatter, boxplot).resolve_scale(
    color='shared'
)

dashboard

### Part 6 - Interactive Visualizations

In [ ]:
# Figure 7: Interactive Scatter Plot with Legend Filtering
legend_selection = alt.selection_point(fields=['Continent'], bind='legend')

# Create the interactive scatter plot
interactive_scatter = alt.Chart(df).mark_circle(size=80).add_params(
    legend_selection
).encode(
    x=alt.X('GDP_per_Capita:Q', title='GDP per Capita'),
    y=alt.Y('Happiness_Score:Q', title='Happiness Score'),
    color=alt.Color(
        'Continent:N',
        title='Continent',
        scale=alt.Scale(scheme='category10')
    ),
    opacity=alt.condition(legend_selection, alt.value(0.8), alt.value(0.1)),
    stroke=alt.condition(legend_selection, alt.value('black'), alt.value('transparent')),
    strokeWidth=alt.condition(legend_selection, alt.value(1), alt.value(0)),
    tooltip=[
        alt.Tooltip('Country:N', title='Country'),
        alt.Tooltip('Continent:N', title='Continent'),
        alt.Tooltip('Happiness_Score:Q', title='Happiness Score', format='.2f'),
        alt.Tooltip('GDP_per_Capita:Q', title='GDP per Capita', format='.2f'),
        alt.Tooltip('Rank:O', title='Global Rank')
    ]
).properties(
    title='Interactive Scatter Plot: GDP vs Happiness by Continent (Click legend to filter)',
    width=500,
    height=400
)

interactive_scatter

In [ ]:
# Figure 8: Choropleth map
# Builds a layered map with sphere/graticule and joins country names then WHR metrics.
# Colors countries by `Happiness_Score` and shows tooltips for country and metrics.

from vega_datasets import data


# Background layers
sphere = alt.sphere()
graticule = alt.graticule()
countries = alt.topo_feature(data.world_110m.url, 'countries')
idname_url = "https://raw.githubusercontent.com/KoGor/Map-Icons-Generator/master/data/world-110m-country-names.tsv"


choropleth = (
    alt.Chart(countries)
    .mark_geoshape(stroke='black', strokeWidth=0.2)
    # 1) id -> name (from Natural Earth)
    .transform_lookup(
        lookup='id',
        from_=alt.LookupData(
            alt.UrlData(idname_url, format=alt.DataFormat(type='tsv')),
            'id',
            ['name']
        )
    )
    # 2) name -> happiness + GDP (from cleaned DataFrame)
    .transform_lookup(
        lookup='name',
        from_=alt.LookupData(df, 'Country', ['Happiness_Score', 'GDP_per_Capita'])
    )
    .encode(
        color=alt.Color(
            'Happiness_Score:Q',
            title='Happiness Score',
            scale=alt.Scale(scheme='plasma')
        ),
        tooltip=[
            alt.Tooltip('name:N', title='Country'),
            alt.Tooltip('Happiness_Score:Q', title='Score', format='.2f'),
            alt.Tooltip('GDP_per_Capita:Q', title='GDP per Capita', format='.3f'),
        ]
    )
)

alt.layer(
    alt.Chart(sphere).mark_geoshape(fill='lightblue'),
    alt.Chart(graticule).mark_geoshape(stroke='white', strokeWidth=0.2),
    choropleth
).project('naturalEarth1'
).properties(width=700, height=500
).configure_view(stroke=None).interactive()

In [ ]:
# Figure 9: Choropleth with selectable metric
# Introduces a parameter bound to a radio control to switch displayed metric.
from vega_datasets import data

# DataFrame with canonical country names (ensure these columns exist)
metrics = [
    'Happiness_Score',
    'Freedom',
    'GDP_per_Capita',
    'Social_Support',
    'Healthy_Life_Expectancy',
    'Generosity',
    'Trust_in_Government'
]

# Background layers
sphere = alt.sphere()
graticule = alt.graticule()
countries = alt.topo_feature(data.world_110m.url, 'countries')

# id -> name mapping URL (fallback provided)
idname_url = "https://raw.githubusercontent.com/KoGor/Map-Icons-Generator/master/data/world-110m-country-names.tsv"

# Parameter bound to a clickable radio control (use binding_select for a dropdown)
metric_param = alt.param(
    name='Metric',
    value='Happiness_Score',
    bind=alt.binding_radio(options=metrics)  # or: alt.binding_select(options=metrics)
)

choropleth = (
    alt.Chart(countries)
    .mark_geoshape(stroke='black', strokeWidth=0.2)
    # 1) id -> name
    .transform_lookup(
        lookup='id',
        from_=alt.LookupData(
            alt.UrlData(idname_url, format=alt.DataFormat(type='tsv')),
            'id',
            ['name']
        )
    )
    # 2) name -> join ALL metrics (so we can fold)
    .transform_lookup(
        lookup='name',
        from_=alt.LookupData(df, 'Country', metrics)
    )
    # 3) fold metrics -> (metric, value)
    .transform_fold(metrics, as_=['metric', 'value'])
    # 4) keep only the selected metric
    .transform_filter(alt.datum.metric == metric_param)
    .encode(
        color=alt.Color('value:Q', title='Selected metric', scale=alt.Scale(scheme='plasma')),
        tooltip=[
            alt.Tooltip('name:N', title='Country'),
            alt.Tooltip('metric:N', title='Metric'),
            alt.Tooltip('value:Q', title='Value', format='.3f'),
        ]
    )
    .add_params(metric_param)
)

alt.layer(
    alt.Chart(sphere).mark_geoshape(fill='lightblue'),
    alt.Chart(graticule).mark_geoshape(stroke='white', strokeWidth=0.5),
    choropleth
).project('naturalEarth1'
).properties(width=700, height=500
).configure_view(stroke=None).interactive()